PREPROCESAMIENTO DE DATOS

Parte de los datos parecen tener ciertas inconsistencias. Limpiaremos y analizaremos los datos con los siguientes pasos:


In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA

In [2]:
df_consolidado = pd.read_csv("data/consolidado.csv")

C:\Users\Usuario\AppData\Local\Temp\ipykernel_23412\2611210249.py:1: DtypeWarning: Columns (0: addrpct, 1: linecm, 2: detailcm) have mixed types. Specify dtype option on import or set low_memory=False.
  df_consolidado = pd.read_csv("data/consolidado.csv")


In [3]:
# 1. Definimos las columnas a limpiar
columnas_limpiar = ['officrid', 'offshld', 'offverb']

print("Antes de los cambios:")
print(df_consolidado[columnas_limpiar].head(6))

# 2. comparación directa: si es un espacio vacío " ", ponemos "N", si no, "Y"
for col in columnas_limpiar:
    # .strip() elimina espacios accidentales alrededor por si acaso
    df_consolidado[col] = df_consolidado[col].apply(lambda x: 'N' if str(x).strip() == "" else 'Y')

print("\nDespués de los cambios:")
print(df_consolidado[columnas_limpiar].head(6))

for col in columnas_limpiar:
    df_consolidado[col] = df_consolidado[col].astype(str)

Antes de los cambios:
  officrid offshld offverb
0                         
1                S       V
2                S        
3                         
4                         
5        I       S       V

Después de los cambios:
  officrid offshld offverb
0        N       N       N
1        N       Y       Y
2        N       Y       N
3        N       N       N
4        N       N       N
5        Y       Y       Y


In [4]:
# Definimos los grupos de columnas
cols_yn = ['officrid', 'offshld', 'offverb']
cols_u = ['sector', 'trhsloc', 'beat']

# Aplicamos limpieza de Y/N
for col in cols_yn:
    if col in df_consolidado.columns:
        df_consolidado[col] = df_consolidado[col].apply(lambda x: 'N' if str(x).strip() == "" else 'Y')

# Aplicamos limpieza de 'U'
for col in cols_u:
    if col in df_consolidado.columns:
        df_consolidado[col] = df_consolidado[col].apply(lambda x: 'U' if str(x).strip() == "" else x)


for col in (cols_yn + cols_u):
    df_consolidado[col] = df_consolidado[col].astype(str)

In [5]:
# --- PASO 1: Limpieza y Transformación de Altura ---

# Forzamos que ht_feet y ht_inch sean números.
# 'coerce' convierte cualquier letra o espacio sucio en NaN (vacío) para que no falle la división.
df_consolidado['ht_feet'] = pd.to_numeric(df_consolidado['ht_feet'], errors='coerce')
df_consolidado['ht_inch'] = pd.to_numeric(df_consolidado['ht_inch'], errors='coerce')

# Ahora sí, calculamos los metros (llenando los vacíos temporales con 0 para la suma)
df_consolidado['meters'] = (df_consolidado['ht_feet'].fillna(0) + (df_consolidado['ht_inch'].fillna(0) / 12)) * 0.3048

# Eliminamos las columnas viejas
df_consolidado.drop(['ht_feet', 'ht_inch'], axis=1, inplace=True)

# --- PASO 2: Estandarización de Etiquetas (Y/N y U) ---
cols_yn = ['officrid', 'offshld', 'offverb']
cols_u = ['sector', 'trhsloc', 'beat']

for col in cols_yn:
    if col in df_consolidado.columns:
        # Limpiamos espacios y convertimos a Y/N
        df_consolidado[col] = df_consolidado[col].astype(str).str.strip().apply(lambda x: 'N' if x == "" or x == "nan" else 'Y')

for col in cols_u:
    if col in df_consolidado.columns:
        # Limpiamos espacios y ponemos 'U' si está vacío
        df_consolidado[col] = df_consolidado[col].astype(str).str.strip().apply(lambda x: 'U' if x == "" or x == "nan" else x)

print("Limpieza previa completada sin errores de tipo.")

Limpieza previa completada sin errores de tipo.


In [6]:
df_consolidado['datestop'] = df_consolidado['datestop'].astype(str)

# Preparar listas para almacenar los valores de los meses y años
months = []
years = []

# Utilizar un bucle for para procesar cada entrada en la columna datestop
for date in df_consolidado['datestop']:
    # Verificar si la fecha tiene 7 dígitos (mes con un dígito)
    if len(date) == 7:
        # Se añade un '0' al principio de la cadena de fecha
        date = '0' + date

    # Extraer el mes y el año
    month = int(date[0:2])
    year = int(date[4:8])

    # Añadir el mes y año a las listas correspondientes
    months.append(month)
    years.append(year)

# Agregar las listas como columnas nuevas en el DataFrame
df_consolidado['month'] = months
df_consolidado['year'] = years

display(df_consolidado.tail(5))
#eliminamos la columna 'datestop'
df_consolidado = df_consolidado.drop(['datestop'], axis=1)

,Unnamed: 0,year,pct,ser_num,datestop,timestop,recstat,inout,trhsloc,perobs,...,sector,beat,post,xcoord,ycoord,dettypcm,linecm,detailcm,meters,month
11820,449178,2010,60,7062,9262010,45,1,O,H,2.0,...,I,U,,989382,155162,CM,1,20,1.7780,9
11821,362162,2010,75,15797,7302010,10,A,O,P,1.0,...,A,U,,1012757,186018,CM,1,85,1.7018,7
11822,208893,2010,123,830,4302010,1630,A,O,P,3.0,...,E,U,,933868,138600,CM,1,46,1.8288,4
11823,551820,2010,115,13122,11232010,2100,1,O,P,1.0,...,E,U,,1014722,214388,CM,1,85,1.8034,11
11824,6678,2010,14,219,1062010,1406,1,I,T,4.0,...,H,11,,987078,215157,CM,1,23,1.8542,1


In [7]:
# 1. Identificar categóricas (2-99 categorías y tipo object)
categorical_cols = [col for col in df_consolidado.columns
                    if 2 <= df_consolidado[col].nunique() <= 99
                    and df_consolidado[col].dtype == 'object']

# 2. Identificar numéricas clave
numeric_cols = ['month', 'year', 'meters', 'age']

# 3. ELIMINAR DUPLICADOS: Aseguramos que no haya columnas repetidas entre las listas
# Si una columna está en ambas, la dejamos solo en una (usualmente la numérica)
categorical_cols = [c for c in categorical_cols if c not in numeric_cols]

# 4. Crear df_copy_18 asegurando columnas ÚNICAS
selected_columns = list(dict.fromkeys(categorical_cols + numeric_cols)) # Elimina duplicados manteniendo orden
df_copy_18 = df_consolidado[selected_columns].copy()

# 5. Definir las listas para el Pipeline (Sin duplicados)
variables_categoricas = categorical_cols
variables_numericas = numeric_cols

# Convertir a string las categóricas para evitar el error de tipos mezclados
for col in variables_categoricas:
    df_copy_18[col] = df_copy_18[col].astype(str)

print(f"Selección lista. Columnas únicas: {len(df_copy_18.columns)}")

Selección lista. Columnas únicas: 5


In [8]:

preprocessor = ColumnTransformer(
    transformers=[
        # Sub-pipeline para Números
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')), # Técnica de imputación
            ('scaler', StandardScaler())                  # Escalado (vital para PCA)
        ]), variables_numericas),

        # Sub-pipeline para Categorías
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), variables_categoricas)
    ])

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)) # Reducción: conserva el 95% de la varianza explicada
])


X_final = final_pipeline.fit_transform(df_copy_18)

print("Pipeline ejecutado con éxito.")
print(f"Forma de la matriz final: {X_final.shape}")

Pipeline ejecutado con éxito.
Forma de la matriz final: (11825, 4)
